# Parameter‑Efficient Fine‑Tuning (PEFT/LoRA) — Colab-ready

**Goal:** Learned how to fine‑tune large pretrained models efficiently using LoRA (a PEFT method). This notebook covers:
1. Full fine‑tuning of `bert-base-uncased` on AG News (small subset for demo).
2. LoRA adapter fine‑tuning using `peft` and comparison with full fine‑tuning.

Run in Google Colab (GPU recommended). Cells include line-by-line explanations.


## 1) Install dependencies

We install `transformers`, `datasets`, `peft`, and other utilities. Run once per session.

In [1]:
# Install required packages (quiet mode)
!pip install -q transformers datasets accelerate peft scikit-learn
import torch, transformers, datasets
print('torch:', torch.__version__, ' | cuda:', torch.cuda.is_available())


torch: 2.10.0+cu128  | cuda: True


## 2) Imports and helper functions
We define `compute_metrics` and import training utilities.

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from peft import LoraConfig, get_peft_model, TaskType

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': float(accuracy_score(labels, preds)), 'f1_macro': float(f1_score(labels, preds, average='macro'))}


## 3) Load dataset and tokenizer
We use **AG News** (4 classes). For demo speed we will use subsets during training; students can remove `.select(...)` to use full sets.

In [3]:
MODEL = 'bert-base-uncased'
raw = load_dataset('ag_news')
print('Splits:', raw.keys())
tokenizer = AutoTokenizer.from_pretrained(MODEL)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding=False, max_length=256)

tok = raw.map(tokenize, batched=True, remove_columns=['text'])
tok = tok.rename_column('label', 'labels')
tok.set_format('torch', columns=['input_ids','attention_mask','labels'])
collator = DataCollatorWithPadding(tokenizer)
tok


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

... (4 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 7600
    })
})

## 4) Full fine‑tuning (baseline)
We fine‑tune all model parameters. For demonstration we use a subset of 30k train examples and 5k eval; reduce further if necessary.

In [4]:
from transformers import logging
logging.set_verbosity_error()
full_model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=4)
full_args = TrainingArguments(
    output_dir='./lab2_full',
    eval_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=100,
    report_to=[]
)
full_trainer = Trainer(
    model=full_model,
    args=full_args,
    train_dataset=tok['train'].select(range(30000)),
    eval_dataset=tok['test'].select(range(5000)),
    data_collator=collator,
    compute_metrics=compute_metrics
)
import time
t0=time.time(); full_trainer.train(); t1=time.time()
full_metrics = full_trainer.evaluate()
print('Full fine-tune done. Time (s):', round(t1-t0,2))
print(full_metrics)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{'loss': '0.7174', 'grad_norm': '0.9671', 'learning_rate': '1.894e-05', 'epoch': '0.05333'}
{'loss': '0.3496', 'grad_norm': '6.013', 'learning_rate': '1.788e-05', 'epoch': '0.1067'}
{'loss': '0.3231', 'grad_norm': '22.51', 'learning_rate': '1.681e-05', 'epoch': '0.16'}
{'loss': '0.3093', 'grad_norm': '11.93', 'learning_rate': '1.574e-05', 'epoch': '0.2133'}
{'loss': '0.296', 'grad_norm': '8.674', 'learning_rate': '1.468e-05', 'epoch': '0.2667'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.2727', 'grad_norm': '13.49', 'learning_rate': '1.361e-05', 'epoch': '0.32'}
{'loss': '0.2289', 'grad_norm': '1.875', 'learning_rate': '1.254e-05', 'epoch': '0.3733'}
{'loss': '0.2182', 'grad_norm': '21.79', 'learning_rate': '1.148e-05', 'epoch': '0.4267'}
{'loss': '0.2367', 'grad_norm': '3.593', 'learning_rate': '1.041e-05', 'epoch': '0.48'}
{'loss': '0.2408', 'grad_norm': '5.229', 'learning_rate': '9.344e-06', 'epoch': '0.5333'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.2389', 'grad_norm': '24.41', 'learning_rate': '8.277e-06', 'epoch': '0.5867'}
{'loss': '0.2227', 'grad_norm': '2.253', 'learning_rate': '7.211e-06', 'epoch': '0.64'}
{'loss': '0.22', 'grad_norm': '4.732', 'learning_rate': '6.144e-06', 'epoch': '0.6933'}
{'loss': '0.2043', 'grad_norm': '2.628', 'learning_rate': '5.077e-06', 'epoch': '0.7467'}
{'loss': '0.2217', 'grad_norm': '1.623', 'learning_rate': '4.011e-06', 'epoch': '0.8'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.1914', 'grad_norm': '3.831', 'learning_rate': '2.944e-06', 'epoch': '0.8533'}
{'loss': '0.267', 'grad_norm': '0.7545', 'learning_rate': '1.877e-06', 'epoch': '0.9067'}
{'loss': '0.1989', 'grad_norm': '0.9993', 'learning_rate': '8.107e-07', 'epoch': '0.96'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': '0.222', 'eval_accuracy': '0.929', 'eval_f1_macro': '0.9284', 'eval_runtime': '37.05', 'eval_samples_per_second': '135', 'eval_steps_per_second': '4.238', 'epoch': '1'}
{'train_runtime': '791.1', 'train_samples_per_second': '37.92', 'train_steps_per_second': '2.37', 'train_loss': '0.2716', 'epoch': '1'}
{'eval_loss': '0.222', 'eval_accuracy': '0.929', 'eval_f1_macro': '0.9284', 'eval_runtime': '35.79', 'eval_samples_per_second': '139.7', 'eval_steps_per_second': '4.387', 'epoch': '1'}
Full fine-tune done. Time (s): 792.28
{'eval_loss': 0.22199209034442902, 'eval_accuracy': 0.929, 'eval_f1_macro': 0.9283704219077433, 'eval_runtime': 35.7872, 'eval_samples_per_second': 139.715, 'eval_steps_per_second': 4.387, 'epoch': 1.0}


## 5) LoRA (PEFT) fine‑tuning
We now attach LoRA adapters to reduce the number of trainable parameters.
LoRA hyperparameters `r`, `alpha`, and `dropout` control the adapter capacity.

In [5]:
base = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=4)
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=['query', 'key', 'value', 'dense']
)
lora_model = get_peft_model(base, peft_config)
lora_model.print_trainable_parameters()

lora_args = TrainingArguments(
    output_dir='./lab2_lora',
    eval_strategy='epoch',
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=1,
    weight_decay=0.0,
    logging_steps=100,
    report_to=[]
)
lora_trainer = Trainer(
    model=lora_model,
    args=lora_args,
    train_dataset=tok['train'].select(range(30000)),
    eval_dataset=tok['test'].select(range(5000)),
    #tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics
)
t0=time.time(); lora_trainer.train(); t1=time.time()
lora_metrics = lora_trainer.evaluate()
print('LoRA fine-tune done. Time (s):', round(t1-t0,2))
print(lora_metrics)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

trainable params: 1,342,468 || all params: 110,827,784 || trainable%: 1.2113
{'loss': '0.8229', 'grad_norm': '0.6059', 'learning_rate': '0.0001894', 'epoch': '0.05333'}
{'loss': '0.3765', 'grad_norm': '2.071', 'learning_rate': '0.0001788', 'epoch': '0.1067'}
{'loss': '0.3385', 'grad_norm': '7.107', 'learning_rate': '0.0001681', 'epoch': '0.16'}
{'loss': '0.3341', 'grad_norm': '5.06', 'learning_rate': '0.0001574', 'epoch': '0.2133'}
{'loss': '0.3096', 'grad_norm': '3.354', 'learning_rate': '0.0001468', 'epoch': '0.2667'}
{'loss': '0.3131', 'grad_norm': '4.313', 'learning_rate': '0.0001361', 'epoch': '0.32'}
{'loss': '0.2645', 'grad_norm': '5.579', 'learning_rate': '0.0001254', 'epoch': '0.3733'}
{'loss': '0.2508', 'grad_norm': '4.383', 'learning_rate': '0.0001148', 'epoch': '0.4267'}
{'loss': '0.265', 'grad_norm': '2.422', 'learning_rate': '0.0001041', 'epoch': '0.48'}
{'loss': '0.257', 'grad_norm': '1.914', 'learning_rate': '9.344e-05', 'epoch': '0.5333'}
{'loss': '0.2562', 'grad_norm'

## 6) Compare and conclude
We print both metrics and train times to see the trade-offs between full fine-tuning and LoRA.
Students: try varying `r` and `lora_alpha` to see effects on performance and training speed.

In [6]:
print('Full FT metrics:', full_metrics)
print('LoRA   metrics:', lora_metrics)


Full FT metrics: {'eval_loss': 0.22199209034442902, 'eval_accuracy': 0.929, 'eval_f1_macro': 0.9283704219077433, 'eval_runtime': 35.7872, 'eval_samples_per_second': 139.715, 'eval_steps_per_second': 4.387, 'epoch': 1.0}
LoRA   metrics: {'eval_loss': 0.22974906861782074, 'eval_accuracy': 0.9196, 'eval_f1_macro': 0.9188322775717284, 'eval_runtime': 42.6273, 'eval_samples_per_second': 117.296, 'eval_steps_per_second': 3.683, 'epoch': 1.0}


## 7) Homework / Extensions
- Try LoRA on a bigger model (e.g., `roberta-base`) and compare.
- Use PEFT + quantization (8-bit) for even lower memory usage.
- Implement a simple script to save and load only LoRA adapter weights.
